# SatQuery AI — Qwen2-VL QLoRA Fine-Tuning Notebook

This notebook fine-tunes **Qwen2-VL-7B-Instruct** on remote sensing benchmark datasets (**VRSBench**, **RSVQA**, **CDVQA**, **BigEarthNet-v2.0**) using 4-bit QLoRA on Google Colab or Kaggle (Free T4/A100 GPU).

## 1. Install Required Libraries

In [ ]:
!pip install -q -U git+https://github.com/huggingface/transformers.git
!pip install -q -U torch torchvision peft bitsandbytes accelerate qwen-vl-utils datasets

## 2. Load Base Model with 4-bit Quantization

In [ ]:
import torch
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType

model_id = "Qwen/Qwen2-VL-7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

model = Qwen2VLForConditionalGeneration.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)
processor = AutoProcessor.from_pretrained(model_id)
print("Qwen2-VL 4-bit Model Loaded Successfully!")

## 3. Apply QLoRA Adapters

In [ ]:
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

## 4. Run Training & Save Adapter Weights

In [ ]:
# Save fine-tuned adapter weights
output_dir = "./qwen2_vl_satquery_adapter"
model.save_pretrained(output_dir)
processor.save_pretrained(output_dir)
print(f"Fine-tuned LoRA weights saved to {output_dir}")